# ML-03 — Frame Your Lane as an ML Task

[![Open In Colab](https://colab.research.google.com/assets/colab-badge.svg)](https://colab.research.google.com/github/widadfatimakhan/flyrank-internship-ml/blob/main/work/notebooks/w02_ml_task_framing.ipynb?flush_cache=true)

This skeleton is yours to fill. Work the sections **in order** — each one has a one-line hint. Simple words, honest numbers.

> Working with an AI assistant? Tell it to read `skills/README.md` first and load the one skill this assignment names on its card.

## 1. My lane as an ML task (type)

*Classification, clustering, ranking, or scoring — which one, and why?*

**Task type: Ranking / scoring.**

My lane's core question is "which pages should an editor review first?" — this is a
"which ones first" question, not "is this one good or bad" (classification) and not
"what natural groups exist in this data" (clustering). I need every qualifying page to
get a numeric opportunity score, then sort by it so an editor works top-down through
the biggest opportunities first. That maps directly to ranking/scoring, with Precision@K
as the metric type this shape of problem typically uses.

In [9]:
# This cell is for CODE (numbers, a query, a check).
# Write your text answer in the cell ABOVE this one — typing sentences here breaks Run All.

import os, sys, subprocess

IN_COLAB = "google.colab" in sys.modules
REPO_URL = "https://github.com/widadfatimakhan/flyrank-internship-ml"
REPO_DIR = "flyrank-internship-ml"

if IN_COLAB:
    if not os.path.isdir(REPO_DIR):
        subprocess.run(["git", "clone", "--depth", "1", REPO_URL, REPO_DIR], check=True)
    os.chdir(REPO_DIR)

import pandas as pd
df = pd.read_csv("data/raw/content_refresh_anonymized.csv")

# Rebuild the Lane 4 slice (same logic as ML-02)
has_position = df["avg_position"] > 0          # avg_position == 0 means "no data," not rank zero
visible = has_position & (df["impressions_90d"] >= 500)
lane_df = df[visible].copy()

tier_avg_ctr = lane_df.groupby("position_tier")["ctr"].transform("mean")
lane_df["ctr_gap"] = tier_avg_ctr - lane_df["ctr"]

print(f"Lane 4 slice loaded: {len(lane_df):,} rows (visible pages with real position data)")

Lane 4 slice loaded: 16,726 rows (visible pages with real position data)


## 2. Target or proxy

*What would you predict? Where does that label come from — observed outcome or a defined rule?*

**Target/proxy: `ctr_gap` — a defined proxy, not an observed outcome.**

There's no existing label for "this page needs a CTR/content review." I define one myself:
for each visible page with real position data, compare its CTR to the average CTR of other
pages in the same position tier. A positive gap means it's underperforming its peers. This
is a rule *I* defined by comparing groups — not something independently observed
(unlike, say, `trend_direction`, which comes from actual measured traffic change). I'll be
careful never to present `ctr_gap` as ground truth — it's a defensible proxy for "opportunity
size," not a proven outcome.

In [10]:
# This cell is for CODE (numbers, a query, a check).
# Write your text answer in the cell ABOVE this one — typing sentences here breaks Run All.

# How much of the visible, position-having population actually has a positive ctr_gap
# (i.e., is genuinely "worth flagging" under this proxy, vs. already at/above their tier average)?

positive_gap = lane_df[lane_df["ctr_gap"] > 0]
print(f"Pages with a positive ctr_gap (underperforming their tier): {len(positive_gap):,} "
      f"({len(positive_gap)/len(lane_df):.1%} of the {len(lane_df):,}-row lane slice)")

Pages with a positive ctr_gap (underperforming their tier): 10,920 (65.3% of the 16,726-row lane slice)


## 3. Success metric

*One metric you can defend. What number means 'good'?*

**Success metric (for now): mean `ctr_gap` in the top-K vs. a random sample of visible pages.**

Since my target is a proxy I defined, not an independent ground-truth label, I can't yet
compute a true Precision@K (there's no separate "correct answer" to check against). For now,
"good" means: the top-K pages by ctr_gap have a meaningfully larger average gap than a random
sample of equally-visible pages — showing the ranking concentrates real opportunity at the top,
not noise. I'll also check the top-K isn't dominated by a single client, since client size in
this dataset is very uneven (one client alone accounts for ~23% of all rows). A stronger metric
— true Precision@K against "did CTR actually improve after review" — would need a forward-looking
observed outcome I don't have yet; that's a reasonable next step, not something I can claim at this moment.

In [11]:
# This cell is for CODE (numbers, a query, a check).
# Write your text answer in the cell ABOVE this one — typing sentences here breaks Run All.

import numpy as np

K = 50
top_k_gap = lane_df.sort_values("ctr_gap", ascending=False).head(K)["ctr_gap"].mean()
random_sample_gap = lane_df.sample(K, random_state=42)["ctr_gap"].mean()

print(f"Mean ctr_gap in top-{K} by ranking:  {top_k_gap:.4f}")
print(f"Mean ctr_gap in a random {K}-page sample: {random_sample_gap:.4f}")
print(f"Difference: top-K's mean gap is {top_k_gap - random_sample_gap:.4f} higher than the random sample's")

# Also check: is the top-K dominated by one client? (client sizes are very uneven in this dataset)
top_k_clients = lane_df.sort_values("ctr_gap", ascending=False).head(K)["client_id"].value_counts()
print(f"\nClient spread in top-{K}: {top_k_clients.nunique() if hasattr(top_k_clients,'nunique') else len(top_k_clients)} unique clients")
print(top_k_clients.head())

client_share_in_lane = (lane_df["client_id"] == "client_19581e27de").mean()
print(f"\nclient_19581e27de's share of the full lane_df: {client_share_in_lane:.1%}")

Mean ctr_gap in top-50 by ranking:  0.3466
Mean ctr_gap in a random 50-page sample: -0.1162
Difference: top-K's mean gap is 0.4627 higher than the random sample's

Client spread in top-50: 6 unique clients
client_id
client_19581e27de    18
client_3fdba35f04    10
client_6208ef0f77     6
client_f369cb89fc     4
client_4e07408562     3
Name: count, dtype: int64

client_19581e27de's share of the full lane_df: 31.2%


**Result:** the top-50 pages by `ctr_gap` average a gap of 0.3466, while a random 50-page
sample averages **-0.1162** — meaning a random sample actually skews toward pages that
outperform their tier on average, not underperform. This makes sense: most of the mass of
a "gap" distribution sits below the true worst cases. The ~0.46 gap difference between top-K
and random confirms the ranking is genuinely concentrating opportunity at the top, not
picking arbitrary pages.

**Client-concentration check:** the top-50 draws from 6 unique clients, but 36% (18/50)
come from a single client (`client_19581e27de`), roughly proportional to that client's
~23% share of the overall dataset — so it's not wildly over-represented relative to its
size, but it is the single largest contributor. Worth watching: if this pattern holds at
larger K, the ranking could end up mostly serving FlyRank's biggest client rather than
surfacing opportunities across the client base evenly. A capstone refinement could weight
or cap per-client representation in the top-K if editor time should be spread more evenly.

## 4. The unit of analysis, as a real dataframe

*Load your lane's slice and show it: one row = one what?*

**Unit of analysis: one row = one content page's 90-day performance snapshot**, filtered to
pages that get meaningful traffic (≥500 impressions/90d) and have real position data
(avg_position > 0, excluding the "no data" placeholder rows). Each row carries the page's
position tier, actual CTR, and its `ctr_gap` score — shown below sorted highest-gap first,
i.e., the actual ranking this lane's ML task produces.

In [12]:
# This cell is for CODE (numbers, a query, a check).
# Write your text answer in the cell ABOVE this one — typing sentences here breaks Run All.

print(f"Unit of analysis: one row = one content page's 90-day snapshot")
print(f"Lane 4 slice: {len(lane_df):,} rows (visible pages with real position data)\n")

ranked = lane_df.sort_values("ctr_gap", ascending=False)
ranked[["content_id", "client_id", "position_tier", "avg_position", "ctr", "ctr_gap"]].head(10)

Unit of analysis: one row = one content page's 90-day snapshot
Lane 4 slice: 16,726 rows (visible pages with real position data)



,content_id,client_id,position_tier,avg_position,ctr,ctr_gap
6004,content_97a08f7b106c,client_19581e27de,top_3,1.5,0.0,0.346572
28651,content_6e6694b67eb6,client_19581e27de,top_3,1.7,0.0,0.346572
5377,content_fb4884e34467,client_4e07408562,top_3,1.5,0.0,0.346572
24076,content_4bb993e9270e,client_6208ef0f77,top_3,1.3,0.0,0.346572
25422,content_7d5ad3f9feee,client_19581e27de,top_3,2.7,0.0,0.346572
9294,content_04c10b7d4b2b,client_19581e27de,top_3,2.5,0.0,0.346572
15315,content_c7b87d5dd8a0,client_19581e27de,top_3,1.9,0.0,0.346572
12807,content_6dd1153d206b,client_7f2253d7e2,top_3,2.3,0.0,0.346572
29465,content_8cbda3f608d0,client_349c41201b,top_3,1.9,0.0,0.346572
26771,content_56c54a03ece2,client_19581e27de,top_3,2.7,0.0,0.346572


In [13]:
# Checking a limitation noticed in the ranking above: the top-10 all show ctr=0
# with an identical ctr_gap. Is this a fluke, or does it affect a lot of the ranking?

tied_at_max = (lane_df["ctr_gap"] == lane_df["ctr_gap"].max())
print(f"Pages tied at the single highest ctr_gap value overall: {tied_at_max.sum():,}")

zero_ctr = lane_df[lane_df["ctr"] == 0]
print(f"\nTotal pages with ctr == 0: {len(zero_ctr):,} ({len(zero_ctr)/len(lane_df):.1%} of the lane slice)")
print("\nImpressions distribution among these zero-CTR pages:")
print(zero_ctr["impressions_90d"].describe())

Pages tied at the single highest ctr_gap value overall: 55

Total pages with ctr == 0: 2,538 (15.2% of the lane slice)

Impressions distribution among these zero-CTR pages:
count      2538.000000
mean       1578.289204
std        4922.865569
min         500.000000
25%         669.250000
50%         930.000000
75%        1490.250000
max      208678.000000
Name: impressions_90d, dtype: float64


**Limitation found while inspecting the ranking.** Every page in the top-10 above has
`ctr=0` and the identical `ctr_gap` value — because the gap can't exceed `tier_avg_ctr - 0`,
every zero-CTR page in a tier ties for that tier's maximum possible score. This means my
current ranking isn't a fine-grained ordering at the very top — it's a large tied block, and
`.head(K)` shows an arbitrary slice of it (whatever order the dataframe happens to be in),
not necessarily the K most urgent pages. A page with 50,000 impressions and 0 clicks is a much
stronger, more confident signal than one with exactly 501 impressions and 0 clicks — but my
current gap treats them identically. **A refinement worth exploring:** weight the gap by
impression volume (or use a confidence-adjusted score) so the ranking can distinguish within
this tied group, rather than treating a barely-qualifying page the same as a clearly-flagged one.

**This connects back to Section 3's metric check.** The top-50 used to validate the success
metric is itself affected by the tie problem found here (Section 4) — an unknown number of
those 50 rows were selected arbitrarily among tied ctr=0 pages, not because they were more
urgent than other tied pages left just outside the top-50. The 0.4627 gap advantage over a
random sample is still real and meaningful, but the *specific 50 pages* in that top-50 aren't
a stable, well-defined set until the ranking is refined with confidence-weighting.

## 5. Why ML beats a fixed rule here

*What makes the pattern too messy for an if-statement?*

A flat CTR threshold (e.g., "flag anything under 1% CTR") can't work here because "good CTR"
depends entirely on position — 2% CTR is excellent at position 20 but poor at position 3. A
single if-statement threshold would misfire constantly: flagging well-performing low-position
pages while missing genuinely underperforming top-position ones. My approach instead compares
each page only to others in its own position tier — a comparison that must be computed from
the current data (group averages), not hardcoded, and would need recalculating if typical
CTR-by-position patterns shifted over time (e.g. after a search algorithm update). That
data-dependent, relative comparison is what makes this an analysis/ML problem rather than a
fixed rule.

In [14]:
# This cell is for CODE (numbers, a query, a check).
# Write your text answer in the cell ABOVE this one — typing sentences here breaks Run All.

# Simulate a flat-threshold rule (e.g., "flag anything under 1% CTR") and compare
# how differently it flags pages vs. the tier-relative ctr_gap approach.

flat_rule_flag = lane_df["ctr"] < 0.01  # naive fixed threshold, ignoring position

print("How many pages does a flat 'ctr < 1%' rule flag, by position tier:")
print(lane_df[flat_rule_flag]["position_tier"].value_counts())
print("\nHow many pages does the tier-relative ctr_gap > 0 approach flag, by position tier:")
print(lane_df[lane_df["ctr_gap"] > 0]["position_tier"].value_counts())

How many pages does a flat 'ctr < 1%' rule flag, by position tier:
position_tier
page_3_5    1059
striking     667
page_1       491
deep         266
top_3         55
Name: count, dtype: int64

How many pages does the tier-relative ctr_gap > 0 approach flag, by position tier:
position_tier
page_1      4515
striking    2966
page_3_5    2849
top_3        295
deep         295
Name: count, dtype: int64


**Result:** the flat threshold rule flags only 491 `page_1` pages, while the tier-relative
approach flags 4,515 — a ~9x difference. This is the failure mode predicted above: at the
top position tier, CTR is naturally high enough that almost no page dips under a flat 1%
cutoff, even when a page is significantly underperforming its (higher-CTR) peers at that
same tier. The flat rule would silently miss the large majority of real opportunities at
the top of the rankings — exactly where fixing a page matters most.

**Note on this framing's current limitations (to revisit before the capstone):**
1. The tied-max-score issue at ctr=0 (see Section 4) means the top of the ranking needs
   confidence-weighting, not just the raw gap, to be truly fine-grained.
2. `ctr_gap` is computed using ALL pages in a tier, including underperformers themselves —
   worth checking whether this makes the "average" a moving target that's partly defined by
   the very pages I'm trying to flag against it.

## Self-check

Before you submit, confirm each line honestly:

- [ ] Every section above is filled — markdown thinking AND the code that backs it
- [ ] The notebook runs top to bottom with no errors (Runtime → Run all)
- [ ] No client names, URLs, or private queries anywhere
- [ ] My claims use careful words: observed, measured, directional, decision-support
- [ ] Committed to my repo under `work/notebooks/` — then submit your repo URL on the card. Done.